# 🎁 Décorateurs

Les décorateurs sont un moyen puissant et flexible de modifier le comportement des fonctions ou des classes en Python.
- Un décorateur est une fonction qui prend une autre fonction comme argument, étend son comportement et retourne une nouvelle fonction.
- Ils sont souvent utilisés pour la journalisation (le *logging*), le contrôle d'accès, la mise en cache et l'instrumentation.
- En Python, les décorateurs sont implémentés en utilisant la syntaxe `@nom_du_decorateur` au-dessus de la définition de la fonction.

In [ ]:
def changecase(func):
  def myinner():
    return func().upper()
  return myinner

@changecase
def myfunction():
  return "Hello John Doe!"

@changecase
def otherfunction():
  return "I am batman!"

print(myfunction())
print(otherfunction())

La fonction décorateur définit généralement une fonction interne qui enveloppe la fonction originale, ajoutant la fonctionnalité désirée avant ou après son appel.
  
## 📦 Portée des variables dans les décorateurs

Pourquoi pouvons-nous accéder à des variables externes telles que `func()` à l'intérieur de la fonction interne ?
Lorsque Python recherche un nom de variable, il suit la règle LEGB :

- Local – à l'intérieur de la fonction

- Enclosing (Englobant) – à l'intérieur de toute fonction externe

- Global – au niveau du module

- Builtins (Intégrés)

## 🔄 Comprendre comment les arguments sont passés

L'exemple précédent n'avait pas besoin d'arguments. Mais que faire si nous voulons décorer une fonction qui **prend** des arguments ?

### 🏷️ Nom de fonction et décorateurs

Lorsque vous écrivez :
```python
@my_decorator
def my_function(x, y):
    return x + y
```

Python fait **automatiquement** ceci en coulisses :
```python
def my_function(x, y):
    return x + y

my_function = my_decorator(my_function)  # ← Le nom pointe maintenant vers quelque chose de nouveau !
```

Donc quand quelqu'un appelle `my_function(3, 5)`, il n'appelle **pas** la fonction originale - il appelle ce que le décorateur a retourné.

### 📝 Exemple étape par étape avec arguments

Créons un décorateur pour une fonction qui prend des arguments spécifiques :

In [ ]:
import time

def timing_decorator(func):
    """A decorator that measures execution time - works with ANY function!"""
    
    def wrapper(*args, **kwargs):  # ← Accept any combination of arguments
        start_time = time.time()
        print(f"Arguments: args={args}, kwargs={kwargs}")
        
        result = func(*args, **kwargs)  # ← Unpack and pass them all to the original
        
        end_time = time.time()
        print(f"Execution time of {func.__name__}: {end_time - start_time:.4f} seconds")
        return result
    
    return wrapper

@timing_decorator
def example_function(n):
    total = 0
    for i in range(n):
        total += i
    return total

@timing_decorator
def another_function(x, y=10, z=20):
    time.sleep(1)
    return x * x + y * z

# Test both functions
example_function(1_000_000)
print("\n" + "="*50 + "\n")
another_function(12, z=3, y=5)

### 🔍 Que se passe-t-il, étape par étape ?

1. Quand nous appelons :
   `another_function(12, z=3, y=5)`

2. Python appelle en réalité :
   `wrapper(12, z=3, y=5)`
   
(parce que `another_function` pointe maintenant vers `wrapper`)

3. À l'intérieur de wrapper :
   - `args` devient `(12,)` - un tuple avec l'argument positionnel
   - `kwargs` devient `{'z': 3, 'y': 5}` - un dict avec les arguments nommés

4. wrapper appelle : `func(*args, **kwargs)` qui se décompresse en `func(12, z=3, y=5)`
   - Ceci appelle la fonction originale `another_function` avec tous les arguments préservés

5. Retourne : Le résultat retourne à travers wrapper jusqu'à vous

Le motif `*args` et `**kwargs` est un "attrape-tout" qui capture tous les arguments passés, puis les transmet exactement tels qu'ils étaient.